# Nemotron Phase 2: Merge Search + Submit

**Phase 2: Tactical adapter exploration.** Evaluates all merge experiments from `data/merge_experiments.csv` against the 600-row val split. Ranks blends by accuracy, packages the best as `submission.zip`.

**Attach:** this code dataset, the base model. All merge sources (Tinker, kienngx variants) must be published on Kaggle Models.


In [ ]:
# Set memory env BEFORE torch import (cannot be changed after).
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Glob-discover instead of hardcoding — Kaggle's path conventions vary
# (`/kaggle/input/datasets/<owner>/<slug>/` for our newer dataset,
#  `/kaggle/input/<slug>/` for older public ones).
import glob, sys
matches = glob.glob("/kaggle/input/**/blackwell_setup.py", recursive=True)
assert matches, "code dataset not attached (no blackwell_setup.py found under /kaggle/input)"
CODE_ROOT = os.path.dirname(matches[0])
print(f"CODE_ROOT = {CODE_ROOT}")
sys.path.insert(0, CODE_ROOT)

import nemotron_local_cv  # noqa: F401
import nemotron_train  # noqa: F401
print("nemotron_local_cv + nemotron_train imported.")


In [ ]:
# Pre-model-load Blackwell setup. The mamba_ssm stub MUST be installed before
# AutoModelForCausalLM.from_pretrained loads modeling_nemotron_h.py.
from blackwell_setup import (  # noqa: E402
    set_cuda_alloc_env,
    set_blackwell_ptxas_env,
    install_mamba_ssm_stub,
)
set_cuda_alloc_env()
set_blackwell_ptxas_env()
install_mamba_ssm_stub()
print("CUDA alloc env + Blackwell ptxas env + mamba_ssm stub ready.")


In [ ]:
# peft + vllm for adapter loading/inference. Some Kaggle images include vllm,
# others do not; install from offline packages to keep this kernel deterministic.
# Internet must be OFF for competition compliance, so `--no-index --find-links`
# is mandatory.
import glob, subprocess, sys
matches = glob.glob("/kaggle/input/**/offline_packages", recursive=True)
assert matches, "offline_packages dir not found under /kaggle/input"
OFFLINE_DIR = matches[0]
print(f"OFFLINE_DIR = {OFFLINE_DIR}")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "--no-cache-dir",
    "--no-index", "--find-links", OFFLINE_DIR,
    "peft", "vllm",
])
print("peft + vllm installed from offline packages.")

# Compatibility shim for newer protobuf runtimes where MessageFactory.GetPrototype
# was removed (some Kaggle/papermill-side deps still call it).
from google.protobuf import message_factory as _message_factory
if not hasattr(_message_factory.MessageFactory, "GetPrototype"):
    def _get_prototype(self, descriptor):
        return _message_factory.GetMessageClass(descriptor)
    _message_factory.MessageFactory.GetPrototype = _get_prototype
    print("Applied protobuf MessageFactory.GetPrototype compatibility shim.")


In [ ]:
from pathlib import Path

# Use the kernel-metadata.json-attached mount directly (faster than kagglehub.download).
BASE_MODEL_PATH = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
assert BASE_MODEL_PATH.is_dir(), f"base model not mounted at {BASE_MODEL_PATH}"
print(f"BASE_MODEL_PATH: {BASE_MODEL_PATH}")
print("  files:", sorted(p.name for p in BASE_MODEL_PATH.iterdir())[:6], "...")

# val_split_600.csv ships inside the code dataset (see scripts/sync_code_to_kaggle.py).
VAL_SPLIT = Path(CODE_ROOT) / "data" / "val_split_600.csv"
assert VAL_SPLIT.exists(), VAL_SPLIT


In [ ]:
# Evaluate all merge experiments from merge_experiments.csv.
# This is the "Phase 2: Merge Search" kernel — runs full CV on all blends.
import sys
from pathlib import Path
import pandas as pd
from nemotron_local_cv import __main__ as cv_cli

SCRIPTS_ROOT = Path(CODE_ROOT) / "scripts"
if SCRIPTS_ROOT.is_dir():
    sys.path.insert(0, str(SCRIPTS_ROOT))
from materialize_merge_experiments import materialize_merge_experiments

merge_plan = Path(CODE_ROOT) / "data" / "merge_experiments.csv"
assert merge_plan.exists(), f"merge_experiments.csv not found at {merge_plan}"

merged_root = Path("/kaggle/working/merged_adapters_phase2")
merged_manifest = Path("/kaggle/working/merged_adapter_manifest_phase2.csv")
print(f"merge plan: {merge_plan}")
materialize_merge_experiments(
    experiments_csv=merge_plan,
    output_root=merged_root,
    manifest_out=merged_manifest,
)

assert merged_manifest.exists(), f"merge manifest not created at {merged_manifest}"

summary_out = Path("/kaggle/working/merge_eval_summary.csv")
debug_dir = Path("/kaggle/working/merge_debug")
if debug_dir.exists():
    import shutil
    shutil.rmtree(debug_dir)

cmd = [
    "nemotron_local_cv",
    "--adapter-manifest",
    str(merged_manifest),
    "--summary-out",
    str(summary_out),
    "--debug-dir",
    str(debug_dir),
    "--debug",
    "--val",
    str(VAL_SPLIT),
    "--model",
    str(BASE_MODEL_PATH),
]
print("$ " + " ".join(cmd))
sys.argv = cmd
rc = cv_cli.main()
assert rc == 0, f"nemotron_local_cv exited with {rc}"

print(f"wrote {summary_out}")


In [ ]:
import pandas as pd
from pathlib import Path

summary = Path("/kaggle/working/merge_eval_summary.csv")
assert summary.exists(), "merge_eval_summary.csv missing"
df = pd.read_csv(summary)
print("All merge evaluations:")
print(df.sort_values("accuracy", ascending=False).to_string(index=False))

best_row = df.loc[df["accuracy"].idxmax()]
best_name = best_row["name"]
best_score = best_row["accuracy"]
print(f"\n✓ Best merge: {best_name} @ {best_score:.4f}")

# Save best name for packaging step
Path("/kaggle/working/best_merge_name.txt").write_text(best_name)


In [ ]:
import zipfile
from pathlib import Path

best_name = Path("/kaggle/working/best_merge_name.txt").read_text().strip()
print(f"Packaging best merge: {best_name}")

merged_root = Path("/kaggle/working/merged_adapters_phase2")
adapter_dir = merged_root / best_name
assert adapter_dir.is_dir(), f"Best merge dir not found: {adapter_dir}"

REQUIRED = ["adapter_config.json", "adapter_model.safetensors"]
missing = [f for f in REQUIRED if not (adapter_dir / f).is_file()]
assert not missing, f"Missing required files in {adapter_dir}: {missing}"

out_zip = Path("/kaggle/working/submission.zip")
out_zip.parent.mkdir(parents=True, exist_ok=True)
if out_zip.exists():
    out_zip.unlink()

with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in REQUIRED:
        zf.write(adapter_dir / fname, arcname=fname)

# Verify
with zipfile.ZipFile(out_zip) as zf:
    names = sorted(zf.namelist())
assert names == sorted(REQUIRED), f"Unexpected zip contents: {names}"

size_gb = out_zip.stat().st_size / 1024**3
print(f"wrote {out_zip}  ({size_gb:.2f} GB)")
print(f"contents: {names}")
print("✓ Ready → click Output > submission.zip > Submit to Competition")
